# <center> <img src="../../labs/img/ITESOLogo.png" alt="ITESO" width="480" height="130"> </center>
# <center> **Departamento de Electrónica, Sistemas e Informática** </center>
---
### <center> **Procesamiento de Datos Masivos** </center>
---
### <center> **Primavera 2025** </center>
---
**Primer Examen**

**Fecha**: 14 de Marzo del 2025

**Nombre del estudiante**: Roberto Osorno Liz 

**Professor**: Pablo Camarillo Ramirez

In [1]:
import findspark
findspark.init()

In [ ]:
from pyspark.sql import SparkSession
# spark://078b2e28e517:7077 cluster original

spark = SparkSession.builder \
    .appName("SparkSQL-Exam-1-Roberto-Osorno-liz") \
    .master("spark://fc91669459e3:7077") \
    .config("spark.ui.port","4041") \
    .getOrCreate()

# Create SparkContext
sc = spark.sparkContext
sc.setLogLevel("ERROR")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/03/14 14:24:55 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
25/03/14 14:24:55 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [3]:
import importlib
import whatsapp2.spark_utils as su

In [4]:
importlib.reload(su)

<module 'whatsapp2.spark_utils' from '/home/jovyan/notebooks/lib/whatsapp2/spark_utils.py'>

In [5]:
columns_info_departments = [
    ("department_id", "integer"),
    ("department_name", "string"),
    ("location", "string")
]


In [6]:
columns_info_employees = [
    ("employee_id", "integer"),
    ("employee_info", "string")
]


In [7]:
schema_departments = su.SparkUtils.generate_schema(columns_info_departments)
schema_employees = su.SparkUtils.generate_schema(columns_info_employees)


In [8]:
import os

# Listar los archivos en el directorio
print(os.listdir("/home/jovyan/notebooks/data/"))

['departments.csv', 'employees.csv', 'exam_P2025_ESI3914B.zip', 'netflix1.csv', 'netflix1.csv.zip']


In [9]:
employee_df = spark \
                .read \
                .schema(schema_employees) \
                .option("header", "true") \
                .csv("/home/jovyan/notebooks/data/employees.csv")

In [10]:
department_df = spark \
                .read \
                .schema(schema_departments) \
                .option("header", "true") \
                .csv("/home/jovyan/notebooks/data/departments.csv")

In [11]:
import json

In [12]:
from pyspark.sql.functions import col, get_json_object, when
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType, DateType

In [13]:
final_employees_df = employee_df \
    .withColumn("name", get_json_object(col("employee_info"), "$.name")) \
    .withColumn("department_id", get_json_object(col("employee_info"), "$.department_id").cast("integer")) \
    .withColumn("salary", get_json_object(col("employee_info"), "$.salary").cast("double")) \
    .withColumn("hire_date", get_json_object(col("employee_info"), "$.hire_date").cast("date"))


In [14]:
final_employees_df.cache()

DataFrame[employee_id: int, employee_info: string, name: string, department_id: int, salary: double, hire_date: date]

In [18]:
final_employees_df = final_employees_df.withColumn(
    "salary_category",
    when(col("salary") >= 55000, "High").otherwise("Low")
)


In [19]:
high_salary_df = final_employees_df.filter(col("salary_category") == "High")
low_salary_df = final_employees_df.filter(col("salary_category") == "Low")

In [21]:
from pyspark.sql.functions import col, avg


In [22]:
avg_high_salary_df = high_salary_df.groupBy("department_id").agg(avg("salary").alias("avg_salary"))
avg_low_salary_df = low_salary_df.groupBy("department_id").agg(avg("salary").alias("avg_salary"))

In [23]:
avg_high_salary_df = avg_high_salary_df.join(department_df, "department_id").select("department_name", "avg_salary")
avg_low_salary_df = avg_low_salary_df.join(department_df, "department_id").select("department_name", "avg_salary")


In [ ]:
avg_high_salary_df.show()

In [ ]:
avg_low_salary_df.show()

In [26]:
top_5_high_salary_df = high_salary_df.orderBy(col("salary").desc()).limit(5)
top_5_low_salary_df = low_salary_df.orderBy(col("salary").asc()).limit(5)

In [27]:
from pyspark.sql.functions import col, datediff, current_date


In [28]:
final_employees_df = final_employees_df.withColumn(
    "years_in_company",
    (datediff(current_date(), col("hire_date")) / 365).cast("integer")
)

In [ ]:
max_years_in_company = final_employees_df.agg({"years_in_company": "max"}).collect()


In [ ]:
employees_with_max_years = final_employees_df.filter(col("years_in_company") == max_years_in_company)
num_employees_with_max_years = employees_with_max_years.count()
